# Typed Path Sampling

This notebook demonstrates PyGraphDB's typed adjacency and sampled path traversal APIs. The example builds a small biomedical graph and samples drug -> protein -> disease paths using edge types.

## Local Development Import

When running from the repository checkout, add `../src` to `sys.path`. If the package is installed, this cell is harmless.

In [ ]:
#@title Installs
INSTALL_PYGRAPHDB=True #@param
if INSTALL_PYGRAPHDB:
  !pip install git+https://github.com/mylonasc/pygraphdb.git
  !pip install lmdb

In [ ]:
import random
import shutil
import tempfile

from pygraphdb.graphdb import Edge, GraphDB, Node
from pygraphdb.kvstores import LMDBStore
from pygraphdb.serializers import PickleSerializer

## Build a Typed Graph

Edge types are stored in `edge.properties['type']`. PyGraphDB uses that value to maintain typed adjacency indexes.

In [ ]:
db_path = tempfile.mkdtemp(prefix='pygraphdb_typed_sampling_')
graph = GraphDB(LMDBStore(path=db_path), PickleSerializer())

drugs = [f'drug-{i}' for i in range(1, 5)]
proteins = [f'protein-{i}' for i in range(1, 9)]
diseases = [f'disease-{i}' for i in range(1, 7)]

for drug_id in drugs:
    graph.put_node(Node(node_id=drug_id, properties={'kind': 'drug'}))
for protein_id in proteins:
    graph.put_node(Node(node_id=protein_id, properties={'kind': 'protein'}))
for disease_id in diseases:
    graph.put_node(Node(node_id=disease_id, properties={'kind': 'disease'}))

edges = []
for drug_index, drug_id in enumerate(drugs):
    for offset in range(4):
        protein_id = proteins[(drug_index * 2 + offset) % len(proteins)]
        edges.append(Edge(
            edge_id=f'{drug_id}-{protein_id}',
            source=drug_id,
            target=protein_id,
            properties={'type': 'drug-to-protein', 'confidence': 0.5 + offset / 10},
        ))

for protein_index, protein_id in enumerate(proteins):
    for offset in range(3):
        disease_id = diseases[(protein_index + offset) % len(diseases)]
        edges.append(Edge(
            edge_id=f'{protein_id}-{disease_id}',
            source=protein_id,
            target=disease_id,
            properties={'type': 'protein-to-disease', 'evidence': 'curated'},
        ))

graph.put_edges_bulk(edges)
len(edges)

## Typed Neighbor Lookup

Typed adjacency avoids loading every incident edge and filtering in Python. The new direction semantics are `out = source -> target`, `in = target -> source`, and `any = both`.

In [ ]:
{
    'drug_1_proteins': [node_id.decode() for node_id in graph.neighbors_by_edge_type('drug-1', 'drug-to-protein', direction='out')],
    'protein_1_drugs': [node_id.decode() for node_id in graph.neighbors_by_edge_type('protein-1', 'drug-to-protein', direction='in')],
    'protein_1_diseases': [node_id.decode() for node_id in graph.neighbors_by_edge_type('protein-1', 'protein-to-disease', direction='out')],
}

## Sample Typed Neighbors

`sample_neighbors` uses reservoir sampling over the typed adjacency iterator, so memory use is bounded by `sample_size`. Pass a seeded RNG for reproducible examples.

In [ ]:
sampled_proteins = graph.sample_neighbors(
    'drug-1',
    'drug-to-protein',
    direction='out',
    sample_size=2,
    rng=random.Random(7),
)

[(record['edge_id'].decode(), record['neighbor_id'].decode()) for record in sampled_proteins]

## Sample Multi-Hop Typed Paths

The pattern below samples up to two `drug-to-protein` edges per drug, then up to two `protein-to-disease` edges per sampled protein.

In [ ]:
seed_drugs = random.Random(11).sample(drugs, 3)
pattern = [
    {'edge_type': 'drug-to-protein', 'direction': 'out', 'sample_size': 2},
    {'edge_type': 'protein-to-disease', 'direction': 'out', 'sample_size': 2},
]

paths = graph.sample_typed_paths(seed_drugs, pattern, rng=random.Random(13))

pretty_paths = []
for sampled_path in paths:
    hop1, hop2 = sampled_path['path']
    pretty_paths.append({
        'drug': sampled_path['seed'].decode(),
        'protein': hop1['target_id'].decode(),
        'disease': hop2['target_id'].decode(),
        'edges': [hop1['edge_id'].decode(), hop2['edge_id'].decode()],
    })

pretty_paths[:10]

## Materialize a Sampled Subgraph

`sample_typed_subgraph` returns the sampled paths plus the full node and edge objects touched by those paths.

In [ ]:
subgraph = graph.sample_typed_subgraph(seed_drugs, pattern, rng=random.Random(13))

{
    'num_paths': len(subgraph['paths']),
    'nodes': sorted(node_id.decode() for node_id in subgraph['nodes']),
    'edges': sorted(edge_id.decode() for edge_id in subgraph['edges']),
}

## Rebuild Typed Adjacency

For existing databases that contain typed edge records but no typed adjacency index, call `rebuild_typed_adjacency()`. It scans stored edges and rebuilds typed adjacency for edges with `properties['type']`.

In [ ]:
rebuilt_count = graph.rebuild_typed_adjacency()
rebuilt_count

## Cleanup

In [ ]:
graph.close()
shutil.rmtree(db_path, ignore_errors=True)